# Daily Animal Review

Everyday animal-level review notebook. It combines the across-session checks and the overall RT/MT/psychometric/JND plots for one animal.

to add:
daily plot - one session in revision

## 1. Setup

In [ ]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "analysis" / "daily_merge.py").exists() and (candidate / "DataFiles").exists():
        ROOT = candidate
        break
else:
    raise FileNotFoundError(
        "Could not find the Mafalda_analysis repo root from the current working directory."
    )

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

ROOT

## 2. Choose Animal

In [ ]:
LINE = "Stakes"
COHORT = "cohort2"
SUBJECT_FILE = "merged_JCS0013.csv"
TRAINING_LEVEL = None
TRAINING_LEVEL_MAX = 16
LOAD_REFERENCE = True

## 3. Load Once, Prepare Both Views

In [ ]:
from Pipeline.daily_plots import (
    load_daily_animal_data,
    load_reference_data,
    prepare_daily_animal_data,
)
from Pipeline.across_sessions import (
    load_change_regions,
    prepare_across_sessions_data,
    subject_id_from_file,
)
import Pipeline.biased_blocks as bb
import pandas as pd


def filter_daily_review_rows(df):
    out = df.copy()
    session_type = pd.to_numeric(out.get("session_type"), errors="coerce")
    short_duration = pd.to_numeric(out.get("short_duration"), errors="coerce")

    keep_non_23 = ~session_type.eq(23)
    keep_23 = pd.Series(False, index=out.index, dtype=bool)
    type23 = out[session_type.eq(23)].copy()
    if not type23.empty:
        labeled_23 = bb.add_biased_block_condition(
            type23,
            biased_session_types=(23,),
            unbiased_rt_session_types=(),
            short_duration_value=None,
        )
        keep_23 = out.index.isin(labeled_23.index[labeled_23["block_condition"].eq("unbiased")])
    keep_23_like_rt_unbiased = keep_23 & short_duration.eq(0)
    return out[keep_non_23 | keep_23_like_rt_unbiased].copy()


df_raw, data_dir = load_daily_animal_data(
    subject_file=SUBJECT_FILE,
    line=LINE,
    cohort=COHORT,
)
df_daily_input = filter_daily_review_rows(df_raw)

subject_id = subject_id_from_file(SUBJECT_FILE)
across_prepared = prepare_across_sessions_data(df_raw, training_level=TRAINING_LEVEL, training_level_max=TRAINING_LEVEL_MAX)
daily_prepared = prepare_daily_animal_data(df_daily_input, training_level=TRAINING_LEVEL, training_level_max=TRAINING_LEVEL_MAX)
regions = load_change_regions(data_dir=data_dir, subject_id=subject_id)
reference = load_reference_data() if LOAD_REFERENCE else None

print(f"Subject: {subject_id}")
print(f"Data directory: {data_dir}")
print(f"Raw rows: {len(df_raw)}")
print(f"Daily-input rows after session-23 unbiased-block RT filter: {len(df_daily_input)}")
print(f"Across-session rows: {len(across_prepared['df'])}")
print(f"Daily-plot rows: {len(daily_prepared['df'])}")
daily_prepared["info"]

## 4. Figure 1: Across-Session Review

Large slide-friendly figure with trial counts on the left and behavioral metrics on the right.

In [ ]:
from Pipeline.across_sessions import plot_across_sessions_combined

fig_across = plot_across_sessions_combined(
    across_prepared,
    regions=regions,
    figsize=(24, 14),
)

## 5. Figure 2: Daily Summary + JND

Large slide-friendly figure with animal info, RT, MT, psychometric, and JND.

In [ ]:
import importlib
import matplotlib.pyplot as plt
import Pipeline.daily_plots as daily_plots

importlib.reload(daily_plots)
plt.close("all")

fig_daily = daily_plots.plot_daily_animal_summary_with_jnd(
    daily_prepared,
    reference=reference,
    figsize=(10, 10),
)

## 6. Figure 3: Timing Histograms

Slide-friendly timing distribution histograms for fixation, CNP, RT, MT, and LNP timings.

In [ ]:
import importlib
import matplotlib.pyplot as plt
import Pipeline.daily_plots as daily_plots

importlib.reload(daily_plots)

fig_hist = daily_plots.plot_timing_histograms(
    daily_prepared,
    subject_id=subject_id,
    figsize=(16, 9),
)

fig_rt_by_abl = daily_plots.plot_rt_histogram_by_abl(
    daily_prepared,
    subject_id=subject_id,
    figsize=(7, 4),
)

## 7. Optional Session Exclusion Report

In [ ]:
from Pipeline.across_sessions import filter_sessions_report

sessions_ok, report, excluded, df_clean = filter_sessions_report(
    across_prepared,
    min_trials=100,
    trials_use="total",
    history_n=10,
    min_prev=5,
    k=3.0,
    min_perf=0.7,
    require_perf_min_completed=20,
)

print("Excluded sessions:")
for line in report:
    print("-", line)

print(f"Clean rows: {len(df_clean)}")

## 8. Optional Save For Google Slides

This writes high-resolution PNG files into `outputs/daily_animal_review/`.

In [ ]:
SAVE_FIGURES = False

if SAVE_FIGURES:
    out_dir = ROOT / "outputs" / "daily_animal_review"
    out_dir.mkdir(parents=True, exist_ok=True)
    fig_across.savefig(out_dir / f"{subject_id}_01_across_sessions.png", dpi=250, bbox_inches="tight")
    fig_daily.savefig(out_dir / f"{subject_id}_02_daily_summary_jnd.png", dpi=250, bbox_inches="tight")
    fig_hist.savefig(out_dir / f"{subject_id}_03_timing_histograms.png", dpi=250, bbox_inches="tight")
    print(out_dir)